# 7.1 對話怎麼表示？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**SFT是不是換成另一種分類架構？

SFT仍在猜下一token，只是訓練資料現在有「誰說話」與「期望回答」。資料格式先分開保存，render才決定邊界。

**把直覺寫成數學：**消息列表→一條序列；問題和答案都進入forward，但不一定都作爲loss目標。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import ByteTokenizer, render_chat

messages = [{"role": "user", "content": "1+1=?"}, {"role": "assistant", "content": "2"}]
x, y = render_chat(messages)
print("X", x.tolist(), "Y", y.tolist())
print("有效目標", y[y != -100].tolist())

**如何讀結果：**同一個TinyLM接口可用；新變化主要是資料與監督位置。

**只改一件事：**只把問題換成2+1，保持格式不變。
